In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from jax import config, vmap

config.update("jax_platforms", "cpu") # disable CUDA
from jax_sph.jax_md import space

In [ ]:
# from matplotlib.colors import LinearSegmentedColormap
# cmap = LinearSegmentedColormap.from_list('own', ['black', "#545046", "#8C670A", "#A88828",  "#939336", "#01AC57", "#2ADB77", "#86DA9E", "#F1F1F1"], 512)

c_data = np.linspace(0, 1, 512)[None, :]
fig, ax = plt.subplots(figsize=(5, 1))
ax.imshow(c_data, cmap="viridis", aspect='auto')
ax.axis('off')
fig.tight_layout()
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.savefig('colorbar.png', dpi=100)

In [ ]:
path = '/local/disk/atoshev/dataset_kolm/datasets/2D_KOLM_4096_140kevery1/test.h5'
data = h5py.File(path, 'r')
traj, t0 = "00000", 5000
r0 = np.asarray(data[f"/{traj}/position"][t0])
u0 = np.asarray(data[f"/{traj}/u"][t0])
r1 = np.asarray(data[f"/{traj}/position"][t0 + 1])
r30 = np.asarray(data[f"/{traj}/position"][t0 + 30])
u30 = np.asarray(data[f"/{traj}/u"][t0 + 30])
r31 = np.asarray(data[f"/{traj}/position"][t0 + 31])
data.close()

discplacement_fn, _ = space.periodic(side=2 * np.pi * np.ones(2))

dt = 0.001
v0 = vmap(discplacement_fn, in_axes=(0, 0))(r1, r0) / dt
v30 = vmap(discplacement_fn, in_axes=(0, 0))(r31, r30) / dt

########################################################################

dx = 0.1
fac = 10
period = 2 * np.pi / fac
offset = 0.03

# top wavy side
xwv1 = np.linspace(-offset, period, 256)
xwv1 = np.concatenate((xwv1, np.ones(50) * period))
xwv1 = np.concatenate((xwv1, np.linspace(period, - offset, 256)))
xwv1 = np.concatenate((xwv1, np.zeros(50) - offset))

ywv1 = 0.02* np.sin(10 * np.linspace(-offset, period, 256)) + period - offset
ywv1 = np.concatenate((ywv1, np.linspace(ywv1[0], ywv1[0] + 0.06, 50)))
ywv1 = np.concatenate((ywv1, 0.02 * np.sin(10 * np.linspace(period, -offset, 256)) + period + offset))
ywv1 = np.concatenate((ywv1, np.linspace(ywv1[-1], ywv1[-1] - 0.06, 50)))

# right wavy side
xwv2 = np.linspace(period - offset, period + offset, 50)
xwv2 = np.concatenate((xwv2, np.sin(10 * np.linspace(0, period, 256)) * 0.02 + period + offset))
xwv2 = np.concatenate((xwv2, np.linspace(xwv2[0], xwv2[0] - 0.0, 50)))
xwv2 = np.concatenate((xwv2, np.sin(10 * np.linspace(period, 0, 256)) * 0.02 + period - offset))

ywv2 = np.zeros(50) - 0.01
ywv2 = np.concatenate((ywv2, np.linspace(0, ywv1[255], 256)))
ywv2 = np.concatenate((ywv2, np.ones(50) * ywv1[255]))
ywv2 = np.concatenate((ywv2, np.linspace(ywv2[-1], 0, 256)))

# fix artefacts in top right corner
xfix = np.array([xwv2[0]+0.001, xwv2[0]+0.2, xwv2[0]+0.2, xwv2[0]+0.001, xwv2[0]+0.001])
yfix = np.array([ywv2[611-256]-0.1, ywv2[611-256]-0.1, ywv2[611-256]+0.5, ywv2[611-256]+0.5, ywv2[611-256]-0.1]) 

xfix2 = np.array([xwv2[0]-0.01, xwv2[0]+0.2, xwv2[0]+0.2, xwv2[0]-0.01, xwv2[0]-0.01])
yfix2 = np.array([ywv2[611-256]+0.001, ywv2[611-256]+0.001, ywv2[611-256]+0.5, ywv2[611-256]+0.5, ywv2[611-256]+0.001])

# left side. Flipping x-y results in bottom side
leftx = np.zeros(256) + 0.003
lefty = np.linspace(0, period - offset, 256)

########################################################################

mask0 = (r0[:, 0] < period+dx/2) & (r0[:, 1] < period+dx/2)
mask30 = (r30[:, 0] < period+dx/2) & (r30[:, 1] < period+dx/2)
rs = [r0[mask0], r30[mask30], r0[mask0], r30[mask30]]
vals = [v0[mask0], v30[mask30], u0[mask0], u30[mask30]]
vmin = min(np.linalg.norm(v, axis=1).min() for v in vals)
vmax = max(np.linalg.norm(v, axis=1).max() for v in vals)

kwargs = dict(cmap="viridis", vmin=vmin, vmax=vmax)
def scatter(var, r_i):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(r_i[:, 0], r_i[:, 1], c=np.linalg.norm(var, axis=1), s=1200, **kwargs)
    ax.fill(xwv1, ywv1, color='white')
    ax.plot(xwv1, ywv1, color='black', lw=4)
    ax.fill(xwv2, ywv2, color='white')
    ax.plot(xwv2, ywv2, color='black', lw=4)
    ax.fill(xfix, yfix, color='white', zorder=10)
    ax.fill(xfix2, yfix2, color='white', zorder=10)
    ax.plot(leftx, lefty, color='black', lw=4)
    ax.plot(lefty, leftx, color='black', lw=4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(0, period)
    ax.set_ylim(0, period)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(f'{name}.png', dpi=100)
    
for i, (var, name, r_i) in enumerate(zip(vals, ['v0', 'v30', 'u0', 'u30'], rs)):
    scatter(var, r_i)